In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
import plotly.graph_objects as go

### We begin by studdying the stability of the trivial equilibrium $\bar{q}^{\circ} = 0$

In [ ]:
def f(tau, xi):
    c = 2*xi
    return (2*c*xi*tau - 1 + np.sqrt((1 - c**2*tau))) / (2*tau)
def g(tau, xi):
    c = 2*xi
    return (2*c*xi*tau - 1 - np.sqrt((1 - c**2*tau))) / (2*tau)

tau_lims = [(0,30.),(0,0.3)]#,(-0.0005,0.003)]
c_list = [0.2,2.0]#,20.0]
omega_lims = [(-3,1.1),(-3,1.1)]#,(-1000,250)]
xi_list = [0.1,1.0]#,10.0]

first_xticks = np.linspace(0,25,6)
second_xticks = np.linspace(0,0.3,7)


lw=2
plane_colors     = ["#6b7280", "#1bde3e"]
BLUE = [[0.0, "#6baed6"], [1.0, "#08306b"]]
RED  = [[0.0, "#fb6a4a"], [1.0, "#67000d"]]
upper_line_color = "#0e4496"
lower_line_color = "#a50318"

fig,axs = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for i, (tau_lim, omega_lim, c, xi) in enumerate(zip(tau_lims, omega_lims, c_list, xi_list)):
    tau_vals = np.linspace(tau_lim[0], tau_lim[1], 1200)
    axs[i].plot(tau_vals, f(tau_vals, xi), color=upper_line_color, label=r'$\omega^\circ_+(\xi,c\tau)$', linewidth=lw)
    axs[i].plot(tau_vals, g(tau_vals, xi), color=lower_line_color, label=r'$\omega^\circ_-(\xi,c\tau)$', linewidth=lw)
    axs[i].set_title(f'c = {c},'+r' $\xi =$' + f'{xi}')
    axs[i].set_xlabel(r'$\tau$')
    if i == 0:
        axs[i].set_ylabel(r'$\omega$')
    axs[i].set_ylim(-3, 1.1)
    axs[i].plot([0, 0], [omega_lim[0], xi**2], color='black', linestyle='-.', linewidth=lw,label='NLS Soliton')
    axs[i].plot([1/c**2, 1/c**2], omega_lim,  color='black', linestyle='-', linewidth=lw,label=r'$1/c^2$')
    #Fill in the region between f and g with gray
    axs[i].fill_between(tau_vals, f(tau_vals, xi), g(tau_vals, xi), where=f(tau_vals, xi)>g(tau_vals, xi),
                                              color=plane_colors[i], alpha=0.5, label='Saddle Region')
    axs[i].legend()

fig.subplots_adjust(wspace=0)
axs[0].set_xticks(first_xticks)
axs[1].set_xticks(second_xticks)

axs[0].set_xlim([-1, 30.])
axs[1].set_xlim([-0.05, 0.3])

#Save as pdf
plt.savefig('Figures/slices_equilibria_stability.pdf')


In [ ]:
import numpy as np
import plotly.graph_objects as go
np.seterr(all="ignore")

# ------------------------- knobs -------------------------
xi_min, xi_max   = 0, 1.1
tau_min, tau_max = 0.01, 5
zlo, zhi         = -3, 1.1
N                = 400
xi_slices        = [0.1, 1.0]

# ---------------------------------------------------------

xi_ceiling = 1/(2*np.sqrt(tau_min)) - 1e-3
if xi_max is None: xi_max = xi_ceiling
xi_max = min(xi_max, xi_ceiling)
xi = np.linspace(xi_min, xi_max, N)
seam    = 1.0 / (4*xi**2)
tau_end = np.minimum(seam, tau_max)
s       = np.linspace(0, 1, N)
T       = tau_min + np.outer(tau_end - tau_min, s)
X       = np.repeat(xi[:, None], N, axis=1)
F, G = f(T, X), g(T, X)

fig = go.Figure()
fig.add_surface(x=T, y=X, z=F, colorscale=BLUE, cmin=zlo, cmax=zhi,
                showscale=False, name="f (upper)")
fig.add_surface(x=T, y=X, z=G, colorscale=RED,  cmin=zlo, cmax=zhi,
                showscale=False, name="g (lower)")

def add_xi_plane(fig, xi_s, plane_color):
    t1 = tau_max
    fig.add_mesh3d(x=[tau_min, t1, t1, tau_min], y=[xi_s]*4, z=[zlo, zlo, zhi, zhi],
                   i=[0, 0], j=[1, 2], k=[2, 3], color=plane_color, opacity=0.6,
                 showlegend=False, hoverinfo="skip", flatshading=True)
for i, xs in enumerate(xi_slices):
    add_xi_plane(fig, xs, plane_colors[i])

fig.update_layout(scene=dict(
        xaxis_title="\u03C4", yaxis_title="\u03BE", zaxis_title="\u03C9",
        zaxis=dict(range=[zlo, zhi]), camera=dict(eye=dict(x=1.7, y=1.2, z=1.2))),
    margin=dict(l=0, r=0, t=5, b=20))

#Save as png
fig.write_image("Figures/equilibria_stability_3d.png", width=500, height=500, scale=2)


### Let us now take a look at the ring equilibria

For now, we're happy by recovering the zero level-set of $W''(r)$ numerically, which is the Hopf bifurcation surface in parameter space. This surface separates the region where $\bar{q}^*$ is a saddle from the region where it is a center.

In [ ]:
from ring_saddle_region import m1_of_tau, Wpp_from_R
xi =1
c = 2*xi
kappa = -1
tau_array = np.linspace(0.01, 1.2, 100)
vectorized_m1_of_tau = np.vectorize(m1_of_tau)
boundary = vectorized_m1_of_tau(tau_array, xi, c)
boundary /= kappa # Convert m to R

#Also check with colorplot
R_array = np.linspace(0, 100, 100)
Tau_grid,R_grid = np.meshgrid(tau_array, R_array, indexing='ij')
Wpp_vec = np.vectorize(Wpp_from_R)
Wpp_grid = Wpp_vec(R_grid, Tau_grid, xi=xi, c=c, kappa=kappa)[0]
plt.figure()
plt.pcolormesh(Tau_grid, R_grid, Wpp_grid, shading='auto', cmap='RdBu',vmin = -10, vmax = 10)
plt.colorbar(label=r'$W^{\prime\prime}(R)$')

plt.plot(tau_array, boundary, color='white', linewidth=2, label=r'$W^{\prime\prime}(R)=0$')
plt.plot([1./c**2]*len(tau_array), np.linspace(0, 100, len(tau_array)), color='white', linewidth=2, linestyle='--', label=r'$R=1/c^2$')
plt.ylabel(r'$R$')
plt.xlabel(r'$\tau$')

In [ ]:
np.seterr(all="ignore")

# ------------------------- data -------------------------
kappa = -1
tau_array = np.linspace(0.01, 5.0, 200)
xi_array  = np.linspace(0.01, 1.5, 200)
Tau_grid, Xi_grid = np.meshgrid(tau_array, xi_array, indexing='ij')
C_grid  = 2*Xi_grid
Surface = vectorized_m1_of_tau(Tau_grid, Xi_grid, C_grid)
heteroclinic_surface = Surface / kappa

zlo, zhi = -0.02, 2#float(finite.min()), float(finite.max())
buffer = 0.  # extra space above the surface for the asymptotic surface

# heteroclinic_surface = np.minimum(heteroclinic_surface, zhi)  # clip the surface to avoid extreme values


# seam is tau = 1/C^2  (== 1/(4 xi^2), same seam as the original routine)
# masks along the tau axis (axis 0 under indexing='ij')
low_mask  = Tau_grid < 1./C_grid**2   # below seam -> RED
high_mask = Tau_grid > 1./C_grid**2   # above seam -> BLUE

# grow the low (red) region one cell UP across the seam so its top edge
# meets the bottom edge of the high (blue) region on a shared grid row.
low_ext = low_mask.copy()
low_ext[1:, :] |= low_mask[:-1, :]

surface_low  = np.where(low_ext,   heteroclinic_surface, np.nan)  # -> RED
surface_high = np.where(high_mask, heteroclinic_surface, np.nan)  # -> BLUE

xi_curve = 1./(4*xi_array**2)
z_component_curve = vectorized_m1_of_tau(xi_curve, xi_array, 2*xi_array)/kappa


# ------------------------- knobs ------------------------
# z / colour range derived from the data (the old -3, 1.1 was tuned for omega).
finite   = heteroclinic_surface[np.isfinite(heteroclinic_surface)]
R_array = np.linspace(zlo, zhi+buffer, 100)
R_grid, Xi_grid2 = np.meshgrid(R_array, xi_array, indexing='ij')
asymptotic_surface = 1./(2*Xi_grid2**2)

xi_slices = [0.1, 1.0]          # both inside the new [0.01, 2.0] xi range

# colour scales / plane colours -- swap for your existing BLUE / RED / plane_colors
BLACK = [[0.0, "#000000"], [0.5, "#000000"], [1.0, "#000000"]]
# RED  = [[0.0, "#4d0b0b"], [0.5, "#c93a3a"], [1.0, "#ffb0b0"]]
BLUE = [[0.0, "#6baed6"], [1.0, "#08306b"]]
RED  = [[0.0,"#67000d" ], [1.0, "#fb6a4a"]]
plane_colors     = ["#6b7280", "#1bde3e"]
# --------------------------------------------------------

fig = go.Figure()

fig.add_surface(x=asymptotic_surface, y=Xi_grid2, z=R_grid, colorscale=BLACK, cmin=zlo, cmax=zhi+buffer,
                showscale=False, name="Asymptotic bound for saddle region (black)")
fig.add_surface(x=Tau_grid, y=Xi_grid, z=heteroclinic_surface,  colorscale=RED,  cmin=zlo, cmax=zhi,
                showscale=False, name="Lower bound for saddle region (red)")

#plot xi_curve on top of the surface
fig.add_scatter3d(x=xi_curve, y=xi_array, z=z_component_curve, mode='lines', line=dict(color='white', width=8 ), showlegend=False)

def add_xi_plane(fig, xi_s, plane_color):
    t0, t1 = tau_array[0], tau_array[-1]
    fig.add_mesh3d(x=[t0, t1, t1, t0], y=[xi_s]*4, z=[zlo, zlo, zhi, zhi],
                   i=[0, 0], j=[1, 2], k=[2, 3], color=plane_color, opacity=0.6,
                   showlegend=False, hoverinfo="skip", flatshading=True)
for i, xs in enumerate(xi_slices):
    add_xi_plane(fig, xs, plane_colors[i])

rot = 2
fig.update_layout(scene=dict(
        xaxis_title="\u03C4", yaxis_title="\u03BE", zaxis_title="R",
        zaxis=dict(range=[zlo, zhi+buffer]), camera=dict(eye=dict(x=-1.25*rot**(-1./2.), y=-1.25*rot**(-1./2.), z=rot*1.25))),
    margin=dict(l=0, r=0, t=5, b=20))

#Set x,y,z limits
fig.update_layout(scene=dict(
    xaxis=dict(range=[tau_array[0], tau_array[-1]]),
    yaxis=dict(range=[xi_array[0], xi_array[-1]]),
    zaxis=dict(range=[zlo, zhi+buffer])
))

#Save as png
fig.write_image("Figures/equilibria_stability_3d_hetero.png", width=500, height=500, scale=2)


In [ ]:
#2D slices now

tau_lims = [(0,50.),(0,0.5)]#,(-0.0005,0.003)]
c_list = [0.2,2.0]#,20.0]
omega_lims = [(0,10),(0,10)]#,(-1000,250)]
xi_list = [0.1,1.0]#,10.0]

first_xticks = np.linspace(0,25,6)
second_xticks = np.linspace(0,0.3,7)


lw=2
plane_colors     = ["#6b7280", "#1bde3e"]
BLUE = [[0.0, "#6baed6"], [1.0, "#08306b"]]
RED  = [[0.0, "#fb6a4a"], [1.0, "#67000d"]]
upper_line_color = "#0e4496"
lower_line_color = "#a50318"
min_R = 1.e-7
max_R = 0
for i, (tau_lim, omega_lim, c, xi) in enumerate(zip(tau_lims, omega_lims, c_list, xi_list)):
    tau_vals = np.linspace(tau_lim[0], tau_lim[1], 1200)
    lower_bound = vectorized_m1_of_tau(tau_vals, xi, c,kappa)/kappa
    max_R = max(max_R, np.nanmax(lower_bound))

fig,axs = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for i, (tau_lim, omega_lim, c, xi) in enumerate(zip(tau_lims, omega_lims, c_list, xi_list)):
    tau_vals = np.linspace(tau_lim[0], tau_lim[1], 1200)
    lower_bound = vectorized_m1_of_tau(tau_vals, xi, c,kappa)/kappa
    axs[i].plot(tau_vals,lower_bound, color=lower_line_color, label=r'$W''(R)= 0$', linewidth=lw)
    axs[i].set_title(f'c = {c},'+r' $\xi =$' + f'{xi}')
    axs[i].set_xlabel(r'$\tau$')
    if i == 0:
        axs[i].set_ylabel(r'$R$')
    axs[i].plot([0, 0], [min_R, max_R], color='black', linestyle='-.', linewidth=lw,label='NLS Soliton')
    axs[i].plot([2/c**2, 2/c**2], [min_R, max_R],  color='black', linestyle='-', linewidth=lw,label=r'$2/c^2$')
    axs[i].plot([1/c**2, 1/c**2], [min_R, max_R],  color='white', linestyle='-', linewidth=4)
    #Fill in the region between surfaces with gray
    axs[i].fill_between(tau_vals, max_R, lower_bound, where=max_R>lower_bound,
                                              color=plane_colors[i], alpha=0.5, label='Saddle Region')
    axs[i].legend(loc='lower center')
    axs[i].set_yscale('log')
    axs[i].set_ylim(min_R,max_R )
    axs[i].text(0.47, 0.75, r'$\tau=1/c^2$', transform=axs[i].transAxes, fontsize=12, 
            color='black', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5, edgecolor='black'))

fig.subplots_adjust(wspace=0)
#Save as pdf
plt.savefig('Figures/slices_equilibria_stability_hetero.pdf')